In [12]:
import requests
import pandas as pd
import time

# -----------------------------
# STATE COORDINATES
# -----------------------------
state_coords = {
    "andhra pradesh": (15.9129, 79.7400),
    "assam": (26.2006, 92.9376),
    "bihar": (25.0961, 85.3131),
    "chhattisgarh": (21.2787, 81.8661),
    "gujarat": (22.2587, 71.1924),
    "haryana": (29.0588, 76.0856),
    "karnataka": (15.3173, 75.7139),
    "kerala": (10.8505, 76.2711),
    "madhya pradesh": (22.9734, 78.6569),
    "maharashtra": (19.7515, 75.7139),
    "odisha": (20.9517, 85.0985),
    "punjab": (31.1471, 75.3412),
    "rajasthan": (27.0238, 74.2179),
    "tamil nadu": (11.1271, 78.6569),
    "telangana": (18.1124, 79.0193),
    "uttar pradesh": (26.8467, 80.9462),
    "west bengal": (22.9868, 87.8550),
    "delhi": (28.7041, 77.1025),
    "jammu and kashmir": (33.7782, 76.5762),
    "puducherry": (11.9416, 79.8083)
}

# -----------------------------
# FETCH FUNCTION
# -----------------------------
def get_nasa_monthly(lat, lon):
    url = "https://power.larc.nasa.gov/api/temporal/monthly/point"
    
    params = {
        "parameters": "PRECTOTCORR,T2M,RH2M",
        "community": "AG",
        "longitude": lon,
        "latitude": lat,
        "start": "1997",
        "end": "2022",
        "format": "JSON"
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        print("❌ Failed:", response.status_code)
        return None

    data = response.json()["properties"]["parameter"]

    # -----------------------------
    # CREATE DATAFRAME
    # -----------------------------
    df = pd.DataFrame({
        "YearMonth": list(data["PRECTOTCORR"].keys()),
        "Rainfall": list(data["PRECTOTCORR"].values()),
        "Temperature": list(data["T2M"].values()),
        "Humidity": list(data["RH2M"].values())
    })

    # -----------------------------
    # CLEAN & FIX MONTHS
    # -----------------------------
    df["YearMonth"] = df["YearMonth"].astype(str)

    # Keep only valid months (01–12)
    df = df[df["YearMonth"].str[-2:].astype(int) <= 12]

    # Extract Year and Month
    df["Year"] = df["YearMonth"].str[:4].astype(int)
    df["Month"] = df["YearMonth"].str[4:].astype(int)

    return df


# -----------------------------
# LOOP THROUGH STATES
# -----------------------------
all_states_data = []

for state, (lat, lon) in state_coords.items():
    print("Fetching:", state)

    df = get_nasa_monthly(lat, lon)

    if df is not None:
        df["State"] = state
        all_states_data.append(df)

    time.sleep(1)  # avoid API rate limit

# -----------------------------
# COMBINE ALL STATES
# -----------------------------
climate_monthly_df = pd.concat(all_states_data, ignore_index=True)

# -----------------------------
# SAVE DATA
# -----------------------------
climate_monthly_df.to_csv("NASA-monthlyINDIA.csv",index=False)

print("✅ Data saved successfully!")
print(climate_monthly_df.head())

Fetching: andhra pradesh
Fetching: assam
Fetching: bihar
Fetching: chhattisgarh
Fetching: gujarat
Fetching: haryana
Fetching: karnataka
Fetching: kerala
Fetching: madhya pradesh
Fetching: maharashtra
Fetching: odisha
Fetching: punjab
Fetching: rajasthan
Fetching: tamil nadu
Fetching: telangana
Fetching: uttar pradesh
Fetching: west bengal
Fetching: delhi
Fetching: jammu and kashmir
Fetching: puducherry
✅ Data saved successfully!
  YearMonth  Rainfall  Temperature  Humidity  Year  Month           State
0    199701      1.03        22.79     71.38  1997      1  andhra pradesh
1    199702      0.00        25.80     62.29  1997      2  andhra pradesh
2    199703      0.22        29.84     54.77  1997      3  andhra pradesh
3    199704      1.23        30.84     57.35  1997      4  andhra pradesh
4    199705      0.35        34.52     44.43  1997      5  andhra pradesh


In [13]:
import pandas as pd

# Load datasets
climate_df = pd.read_csv("datasets/NASA-monthlyINDIA.csv")
crop_df = pd.read_csv("datasets/crop_yield(19690).csv")

# Clean text
climate_df["State"] = climate_df["State"].str.lower().str.strip()
crop_df["State"] = crop_df["State"].str.lower().str.strip()
crop_df["Season"] = crop_df["Season"].str.strip()

# Function to assign season + crop year
def assign_season_and_year(row):
    month = row["Month"]
    year = row["Year"]

    # Season mapping
    if month in [6,7,8,9,10]:
        season = "Kharif"
        crop_year = year

    elif month in [11,12]:
        season = "Rabi"
        crop_year = year

    elif month in [1,2,3]:
        season = "Rabi"
        crop_year = year - 1   # 🔥 Rabi year shift

    elif month in [4,5]:
        season = "Summer"
        crop_year = year

    else:
        season = None
        crop_year = year

    return pd.Series([season, crop_year])

# Apply function
climate_df[["Season", "Crop_Year"]] = climate_df.apply(assign_season_and_year, axis=1)

In [14]:
seasonal_climate = climate_df.groupby(
    ["State", "Crop_Year", "Season"]
).agg({
    "Rainfall": "sum",
    "Temperature": "mean",
    "Humidity": "mean"
}).reset_index()

print(seasonal_climate.head())

            State  Crop_Year  Season  Rainfall  Temperature   Humidity
0  andhra pradesh       1996    Rabi      1.25    26.143333  62.813333
1  andhra pradesh       1997  Kharif     22.11    29.072000  70.230000
2  andhra pradesh       1997    Rabi     10.31    26.238000  74.958000
3  andhra pradesh       1997  Summer      1.58    32.680000  50.890000
4  andhra pradesh       1998  Kharif     24.46    28.932000  74.216000


In [15]:
# Clean crop dataset
crop_df["State"] = crop_df["State"].str.lower().str.strip()
crop_df["Season"] = crop_df["Season"].str.strip()

# Optional: Remove "Whole Year" rows (recommended for simplicity)
crop_df = crop_df[crop_df["Season"] != "Whole Year"]

In [16]:
merged_df = pd.merge(
    crop_df,
    seasonal_climate,
    on=["State", "Crop_Year", "Season"],
    how="inner"
)

print(merged_df.head())
print("Final shape:", merged_df.shape)

           Crop  Crop_Year  Season  State     Area  Production  \
0     Arhar/Tur       1997  Kharif  assam   6637.0        4685   
1   Castor seed       1997  Kharif  assam    796.0          22   
2  Cotton(lint)       1997  Kharif  assam   1739.0         794   
3          Gram       1997    Rabi  assam   2979.0        1507   
4          Jute       1997  Kharif  assam  94520.0      904095   

   Annual_Rainfall  Fertilizer  Pesticide     Yield  Rainfall  Temperature  \
0           2051.4   631643.29    2057.47  0.710435     29.17       26.908   
1           2051.4    75755.32     246.76  0.238333     29.17       26.908   
2           2051.4   165500.63     539.09  0.420909     29.17       26.908   
3           2051.4   283511.43     923.49  0.465455      4.45       18.678   
4           2051.4  8995468.40   29301.20  9.919565     29.17       26.908   

   Humidity  
0    81.388  
1    81.388  
2    81.388  
3    61.770  
4    81.388  
Final shape: (11060, 13)


In [17]:
print(merged_df[["State", "Crop_Year", "Rainfall"]].head())
print(merged_df.isnull().sum())

   State  Crop_Year  Rainfall
0  assam       1997     29.17
1  assam       1997     29.17
2  assam       1997     29.17
3  assam       1997      4.45
4  assam       1997     29.17
Crop               0
Crop_Year          0
Season             0
State              0
Area               0
Production         0
Annual_Rainfall    0
Fertilizer         0
Pesticide          0
Yield              0
Rainfall           0
Temperature        0
Humidity           0
dtype: int64


In [ ]:
merged_df.to_csv("crop_yield_withMonthlyCLimateData.csv", index=False)